# KVO_06 v1.1 — U-Net-guided SAM3 Musanga crown extraction on Baego

**Notebook version: `KVO_06 v1.1`**

**Primary observational basis:** six exhaustively digitized crown-validation areas in `ROI_Crowns_Kvo.shp`, with **all visible crowns** digitized in `Crown_Kristof.shp`.

The notebook now separates two tasks:

1. **Observational validation:** within the six crown ROIs, use *all* U-Net candidates passing the fixed probability/coherence criteria, run SAM3, and compare automatic objects against the exhaustive visible-crown reference.
2. **Baego production pilot:** outside this validation logic, the existing ranked top-500 workflow remains available as an optional production-scale pilot.

This makes the six ROIs the observational basis for evaluating the fully automatic **U-Net detection → SAM3 crown delineation** workflow. KVO_03 remains the independent species-classification validation and KVO_05 the prompt-conditioned SAM3 boundary benchmark.


## 0R — Resume a new Colab session

Run **0R**, then **Block 2 only if the SAM3 environment is missing**, then **0S**, then **Blocks 10–13**. Skip Blocks 1 and 3–9. Saved Drive masks are preserved.


In [ ]:
# BLOCK 0R — Fresh runtime resume (no original inference)
import os,sys,subprocess,shutil,warnings
from pathlib import Path
import numpy as np,pandas as pd,geopandas as gpd,rasterio
from rasterio.windows import Window
from rasterio.features import shapes
from shapely.geometry import Point,shape
from PIL import Image
from tqdm.auto import tqdm
from google.colab import drive,userdata
ROOT=Path("/content/drive/MyDrive/Colab Notebooks/Musanga")
if not ROOT.exists():drive.mount("/content/drive")
GEO=ROOT/"data"/"Geo_Ref";AD=GEO/"Annotations"
OUT=ROOT/"KVO_work"/"KVO_06_UNet_guided_SAM3";RAW=OUT/"sam3_raw"
RASTER=GEO/"Baego_georef.tif";PROB=ROOT/"KVO_work"/"KVO_04_final_multisite"/"probability_maps"/"Baego_musanga_probability_uint16.tif"
CROWN_REF=AD/"Crown_Kristof.shp";CROWN_ROI=AD/"ROI_Crowns_Kvo.shp"
CHIP_SIZE_M=24.;MIN_PRED_AREA_M2=2.;MAX_PRED_AREA_M2=500.
manifest=pd.read_csv(OUT/"candidate_manifest.csv")
roi=gpd.read_file(CROWN_ROI)
if roi.crs is None:raise ValueError("ROI missing CRS")
with rasterio.open(RASTER) as src:
    if roi.crs!=src.crs:roi=roi.to_crs(src.crs)
roi_union=roi.geometry.union_all()
SAMPY="/content/sam3env/bin/python";RUNNER=Path("/content/run_sam3_kvo06.py")
env=os.environ.copy();token=userdata.get("HF_TOKEN")
if token:env["HF_TOKEN"]=token
print("Saved candidates:",len(manifest),"existing SAM3 masks:",len(list(RAW.glob("cand_*.npz"))))
print("SAM3 env:",Path(SAMPY).exists())


## 0S — Restore runner without running original SAM3 inference

Recreates only the Python runner script that was previously embedded in Block 4.


In [ ]:
# BLOCK 0S — Runner definition only
RUNNER=Path("/content/run_sam3_kvo06.py")
RUNNER.write_text(r'''import sys
from pathlib import Path
import numpy as np,pandas as pd,torch
from PIL import Image
from tqdm import tqdm
from sam3.model_builder import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor

manifest=Path(sys.argv[1]); outdir=Path(sys.argv[2]); box_m=float(sys.argv[3]); force=sys.argv[4]=="1"
outdir.mkdir(parents=True,exist_ok=True); df=pd.read_csv(manifest)
if not torch.cuda.is_available(): raise RuntimeError("KVO_06 requires CUDA")
torch.backends.cuda.matmul.allow_tf32=True; torch.backends.cudnn.allow_tf32=True
torch.autocast("cuda",dtype=torch.bfloat16).__enter__(); torch.inference_mode().__enter__()
bpe="/content/sam3/sam3/assets/bpe_simple_vocab_16e6.txt.gz"
model=build_sam3_image_model(bpe_path=bpe); processor=Sam3Processor(model)
print("SAM3 ready; candidates=",len(df),flush=True)
for r in tqdm(df.itertuples(),total=len(df),desc="DELINEATE"):
    op=outdir/f"cand_{int(r.candidate_id):05d}.npz"
    if op.exists() and not force: continue
    img=Image.open(r.path).convert("RGB")
    state=processor.set_image(img)
    # Normalized centre-x, centre-y, width, height; candidate is chip centre.
    frac=min(0.95,box_m/(r.width*r.gsd))
    box=np.array([[0.5,0.5,frac,frac]],dtype=np.float32)
    out=processor.add_geometric_prompt(box=box,label=np.array([1],dtype=np.int32),state=state)
    masks=out["masks"].detach().cpu().numpy().astype(np.uint8)
    if masks.ndim==4: masks=masks[:,0]
    scores=out["scores"].detach().float().cpu().numpy()
    np.savez_compressed(op,masks=masks,scores=scores)
print("DONE",flush=True)
''')

print('SAM3 runner restored:',RUNNER)


## 0 — Host environment and configuration

Use a **GPU runtime**. KVO_06 reuses the same isolated official SAM3 setup as KVO_05.


In [ ]:
import sys, subprocess, os, json, random, warnings, gc, shutil, math
from pathlib import Path
print("Host Python:",sys.version)
subprocess.run(["nvidia-smi"],check=False)
!pip -q install -U "numpy>=2,<2.6" "huggingface_hub<2" geopandas rasterio pyogrio shapely scikit-learn matplotlib tqdm pillow


In [ ]:
import numpy as np, pandas as pd, geopandas as gpd, matplotlib.pyplot as plt
import rasterio
from rasterio.windows import Window
from rasterio.features import shapes
from shapely.geometry import shape, Point
from tqdm.auto import tqdm
from PIL import Image
from google.colab import drive
warnings.filterwarnings("ignore")

SEED=42
random.seed(SEED); np.random.seed(SEED)
DRIVE_ROOT=Path("/content/drive"); MYDRIVE=DRIVE_ROOT/"MyDrive"
def healthy():
    try: return MYDRIVE.exists() and any(MYDRIVE.iterdir())
    except OSError: return False
if not healthy(): drive.mount(str(DRIVE_ROOT))

ROOT=MYDRIVE/"Colab Notebooks"/"Musanga"; GEO=ROOT/"data"/"Geo_Ref"; AD=GEO/"Annotations"
OUT=ROOT/"KVO_work"/"KVO_06_UNet_guided_SAM3"
CHIPS=OUT/"chips"; RAW=OUT/"sam3_raw"; VIS=OUT/"examples"
for p in [OUT,CHIPS,RAW,VIS]: p.mkdir(parents=True,exist_ok=True)

RASTER=GEO/"Baego_georef.tif"
PROB=ROOT/"KVO_work"/"KVO_04_final_multisite"/"probability_maps"/"Baego_musanga_probability_uint16.tif"
CROWN_REF=AD/"Crown_Kristof.shp"
CROWN_ROI=AD/"ROI_Crowns_Kvo.shp"
assert RASTER.exists(),RASTER
assert PROB.exists(),PROB
assert CROWN_REF.exists(),CROWN_REF
assert CROWN_ROI.exists(),CROWN_ROI

# Candidate extraction. This is intentionally conservative and computationally cheap.
PROB_THRESHOLD=0.90
SCAN_WINDOW_PX=2048
PEAK_MIN_DISTANCE_M=10.0
CONTEXT_RADIUS_M=4.0       # score support around each peak
MIN_CONTEXT_MEAN=0.60      # reject isolated hot pixels
VALIDATION_MODE=True        # primary KVO_06 observational experiment
MAX_CANDIDATES=500         # used only if VALIDATION_MODE=False
CHIP_SIZE_M=24.0          # local RGB context around each U-Net candidate
BOX_SIZE_M=10.0           # SAM3 geometric prompt centered on probability peak
MIN_PRED_AREA_M2=2.0
MAX_PRED_AREA_M2=500.0
FORCE_CHIPS=False
FORCE_SAM=False
print("OUT:",OUT)


## 1 — Extract coherent U-Net candidates

Candidate detection: probability ≥0.90, local-context support, then 10 m spatial NMS.

**Validation mode (default):** retain every qualifying candidate whose point lies within the six polygons of `ROI_Crowns_Kvo.shp`. There is no top-N confidence cap inside the observational areas. RGB chips and SAM3 are generated only for those candidates.

**v0.8:** the complete Block 1 is resume-safe: candidate tuples are converted to the downstream dataframe/manifest format within the same block, and the ROI shapefile is read with SHX restoration enabled for robust Drive-mounted shapefiles.


In [ ]:
# v0.8 — coherent candidate extraction + ROI-filtered chip/manifest preparation.
# Pass 1: large-window scan; retain the strongest pixel per 10-m metric grid cell.
with rasterio.open(PROB) as ps:
    pcrs=ps.crs; ptr=ps.transform
    gsd=float((abs(ptr.a)+abs(ptr.e))/2)
    grid_m=float(PEAK_MIN_DISTANCE_M)
    nx=math.ceil(ps.width/SCAN_WINDOW_PX); ny=math.ceil(ps.height/SCAN_WINDOW_PX)
    nwin=nx*ny; best={}
    thr=int(round(PROB_THRESHOLD*10000))
    for r0 in tqdm(range(0,ps.height,SCAN_WINDOW_PX),total=ny,desc="U-NET SCAN"):
        for c0 in range(0,ps.width,SCAN_WINDOW_PX):
            hh=min(SCAN_WINDOW_PX,ps.height-r0); ww=min(SCAN_WINDOW_PX,ps.width-c0)
            a=ps.read(1,window=Window(c0,r0,ww,hh))
            rr,cc=np.where((a!=65535) & (a>=thr))
            if len(rr)==0: continue
            vals=a[rr,cc].astype(np.float32)/10000.0
            gr=rr+r0; gc=cc+c0
            xs,ys=rasterio.transform.xy(ptr,gr,gc,offset="center")
            xs=np.asarray(xs); ys=np.asarray(ys)
            gx=np.floor(xs/grid_m).astype(np.int64); gy=np.floor(ys/grid_m).astype(np.int64)
            order=np.argsort(vals)[::-1]; seen=set()
            for j in order:
                key=(int(gx[j]),int(gy[j]))
                if key in seen: continue
                seen.add(key)
                rec=(float(vals[j]),int(gr[j]),int(gc[j]),float(xs[j]),float(ys[j]))
                old=best.get(key)
                if old is None or rec[0]>old[0]: best[key]=rec

    print("Large raster windows scanned:",nwin)
    print("Strong coarse candidate cells:",len(best))

    # Pass 2: measure local support around each strong peak directly from probability map.
    rad=max(1,int(round(CONTEXT_RADIUS_M/gsd)))
    coherent=[]
    for peak,row,col,x,y in tqdm(best.values(),desc="CONTEXT SCORE"):
        w=Window(col-rad,row-rad,2*rad+1,2*rad+1)
        a=ps.read(1,window=w,boundless=True,fill_value=65535)
        v=a[a!=65535].astype(np.float32)/10000.0
        context_mean=float(v.mean()) if len(v) else 0.0
        context_high=float((v>=PROB_THRESHOLD).mean()) if len(v) else 0.0
        if context_mean < MIN_CONTEXT_MEAN: continue
        # Ranking rewards both a strong peak and broad local support.
        score=0.40*peak + 0.45*context_mean + 0.15*context_high
        coherent.append((score,peak,context_mean,context_high,row,col,x,y))

print("Coherent candidates before spatial NMS:",len(coherent))

# Pass 3: exact 10-m NMS using a spatial hash, highest coherent score first.
coherent.sort(key=lambda z:z[0],reverse=True)
accepted=[]; occupied={}
for rec in tqdm(coherent,desc="CANDIDATE NMS"):
    score,peak,cmean,chigh,row,col,x,y=rec
    kx=int(math.floor(x/grid_m)); ky=int(math.floor(y/grid_m)); ok=True
    for dx in (-1,0,1):
        for dy in (-1,0,1):
            for qx,qy in occupied.get((kx+dx,ky+dy),[]):
                if (x-qx)**2+(y-qy)**2 < PEAK_MIN_DISTANCE_M**2:
                    ok=False; break
            if not ok: break
        if not ok: break
    if ok:
        accepted.append(rec)
        occupied.setdefault((kx,ky),[]).append((x,y))

cand=pd.DataFrame(accepted,columns=["candidate_score","peak_prob","context_mean","context_high_frac",
                                    "row","col","x","y"])
cand=cand.sort_values("candidate_score",ascending=False).reset_index(drop=True)
n_before=len(cand)
cg=gpd.GeoDataFrame(cand,geometry=gpd.points_from_xy(cand.x,cand.y),crs=pcrs)

if VALIDATION_MODE:
    # Robust to a temporarily missing/stale .shx view on mounted Google Drive.
    with rasterio.Env(SHAPE_RESTORE_SHX='YES'):
        roi=gpd.read_file(CROWN_ROI)
    if roi.crs is None: raise ValueError("ROI_Crowns_Kvo.shp has no CRS.")
    if roi.crs!=pcrs: roi=roi.to_crs(pcrs)
    roi=roi[roi.geometry.notna() & ~roi.geometry.is_empty].copy()
    roi["geometry"]=roi.geometry.make_valid()
    roi_union=roi.geometry.union_all()
    cg=cg[cg.geometry.intersects(roi_union)].copy()
    print("Validation ROIs:",len(roi))
    print("Qualifying U-Net candidates inside validation ROIs:",len(cg))
else:
    if MAX_CANDIDATES is not None: cg=cg.head(MAX_CANDIDATES).copy()

cg=cg.reset_index(drop=True)
cg["candidate_id"]=np.arange(len(cg),dtype=int)
cg.to_file(OUT/"Baego_UNet_candidates.gpkg",layer="candidates",driver="GPKG")
print("Peak threshold:",PROB_THRESHOLD)
print("Minimum context mean:",MIN_CONTEXT_MEAN)
print("Minimum separation m:",PEAK_MIN_DISTANCE_M)
print("Candidates after NMS:",n_before)
print("Candidates retained for SAM3:",len(cg),"(validation mode)" if VALIDATION_MODE else "(production pilot)")
display(cg[["candidate_id","candidate_score","peak_prob","context_mean","context_high_frac"]].head(10))

# Prepare one compact RGB chip per retained candidate only.
manifest=[]
with rasterio.open(RASTER) as src:
    if src.crs!=pcrs: raise ValueError("RGB and probability raster CRS differ")
    px=float((abs(src.transform.a)+abs(src.transform.e))/2)
    chip_px=max(128,int(round(CHIP_SIZE_M/px)))
    if chip_px%2: chip_px+=1
    for r in tqdm(cg.itertuples(),total=len(cg),desc="RGB CHIPS"):
        row,col=src.index(r.x,r.y)
        w=Window(col-chip_px//2,row-chip_px//2,chip_px,chip_px)
        op=CHIPS/f"cand_{r.candidate_id:05d}.jpg"
        if FORCE_CHIPS or not op.exists():
            rgb=np.moveaxis(src.read([1,2,3],window=w,boundless=True,fill_value=0),0,-1).astype(np.uint8)
            Image.fromarray(rgb).save(op,quality=95)
        manifest.append(dict(candidate_id=r.candidate_id,path=str(op),x=r.x,y=r.y,
                             candidate_score=r.candidate_score,peak_prob=r.peak_prob,
                             context_mean=r.context_mean,context_high_frac=r.context_high_frac,
                             row_off=int(w.row_off),col_off=int(w.col_off),
                             height=chip_px,width=chip_px,gsd=px))
manifest=pd.DataFrame(manifest)
manifest.to_csv(OUT/"candidate_manifest.csv",index=False)
print("RGB chips:",len(manifest)," chip pixels:",chip_px)


## 2 — Isolated official SAM3 environment

In [ ]:
# SAM3 needs numpy<2, while current Colab geospatial packages need numpy>=2.
# Use uv to create a real isolated Python 3.12 environment; do NOT modify host Python.
UV=Path("/content/uv")
SAMENV=Path("/content/sam3env")
SAMREPO=Path("/content/sam3")

if not UV.exists():
    subprocess.run(["bash","-lc","curl -LsSf https://astral.sh/uv/install.sh | env UV_INSTALL_DIR=/content sh"],check=True)
    # uv installer may name the binary directly under /content.
    candidates=[Path("/content/uv"),Path("/content/uv/uv"),Path.home()/".local/bin/uv"]
    UV=next((p for p in candidates if p.exists() and p.is_file()),None)
    assert UV is not None,"uv installation failed"
else:
    UV=Path("/content/uv")
print("uv:",subprocess.check_output([str(UV),"--version"],text=True).strip())

# Recreate only the isolated SAM environment if a previous failed venv is present.
if SAMENV.exists() and not (SAMENV/"bin/python").exists():
    shutil.rmtree(SAMENV)
if not (SAMENV/"bin/python").exists():
    subprocess.run([str(UV),"venv","--python","3.12",str(SAMENV)],check=True)

SAMPY=str(SAMENV/"bin/python")
# Reuse Colab's CUDA-enabled torch wheel via explicit install in the isolated env.
subprocess.run([str(UV),"pip","install","--python",SAMPY,
                "torch==2.10.0","torchvision",
                "--index-url","https://download.pytorch.org/whl/cu128"],check=True)

if not SAMREPO.exists():
    subprocess.run(["git","clone","--depth","1","https://github.com/facebookresearch/sam3.git",str(SAMREPO)],check=True)
else:
    subprocess.run(["git","-C",str(SAMREPO),"pull","--ff-only"],check=True)

# Official SAM3 currently requires numpy>=1.26,<2. Keep that constraint inside this env only.
subprocess.run([str(UV),"pip","install","--python",SAMPY,"setuptools<82"],check=True)
# The image builder imports modules that use packages listed by Meta under
# notebook/dev extras (not only the minimal core dependencies). Install the
# official extras rather than discovering them one-by-one at runtime.
subprocess.run([str(UV),"pip","install","--python",SAMPY,
                "-e",str(SAMREPO)+"[notebooks,dev]"],check=True)

subprocess.run([SAMPY,"-c",
    "import sys,numpy,torch,sam3; print('SAM Python',sys.version.split()[0],'numpy',numpy.__version__,'torch',torch.__version__,'CUDA',torch.cuda.is_available())"],
    check=True)
print("HOST unchanged: Python",sys.version.split()[0],"numpy",np.__version__,"rasterio",rasterio.__version__)


## 3 — Hugging Face authentication

In [ ]:
from google.colab import userdata
try: HF_TOKEN=userdata.get("HF_TOKEN")
except Exception: HF_TOKEN=None
assert HF_TOKEN,"Add your Hugging Face read token to Colab Secrets as HF_TOKEN."
print("HF_TOKEN found (value not displayed).")


## 4 — SAM3 delineation of U-Net candidates

SAM3 is now used as a **local delineator**, not a wall-to-wall detector. Each U-Net candidate supplies a small approximate box centred on the probability peak. SAM3 receives the local RGB chip plus that geometric prompt and returns a crown mask.

This is the computational advantage of the revised workflow: SAM3 is run only where the U-Net already provides evidence for Musanga.


In [ ]:
RUNNER=Path("/content/run_sam3_kvo06.py")
RUNNER.write_text(r'''import sys
from pathlib import Path
import numpy as np,pandas as pd,torch
from PIL import Image
from tqdm import tqdm
from sam3.model_builder import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor

manifest=Path(sys.argv[1]); outdir=Path(sys.argv[2]); box_m=float(sys.argv[3]); force=sys.argv[4]=="1"
outdir.mkdir(parents=True,exist_ok=True); df=pd.read_csv(manifest)
if not torch.cuda.is_available(): raise RuntimeError("KVO_06 requires CUDA")
torch.backends.cuda.matmul.allow_tf32=True; torch.backends.cudnn.allow_tf32=True
torch.autocast("cuda",dtype=torch.bfloat16).__enter__(); torch.inference_mode().__enter__()
bpe="/content/sam3/sam3/assets/bpe_simple_vocab_16e6.txt.gz"
model=build_sam3_image_model(bpe_path=bpe); processor=Sam3Processor(model)
print("SAM3 ready; candidates=",len(df),flush=True)
for r in tqdm(df.itertuples(),total=len(df),desc="DELINEATE"):
    op=outdir/f"cand_{int(r.candidate_id):05d}.npz"
    if op.exists() and not force: continue
    img=Image.open(r.path).convert("RGB")
    state=processor.set_image(img)
    # Normalized centre-x, centre-y, width, height; candidate is chip centre.
    frac=min(0.95,box_m/(r.width*r.gsd))
    box=np.array([[0.5,0.5,frac,frac]],dtype=np.float32)
    out=processor.add_geometric_prompt(box=box,label=np.array([1],dtype=np.int32),state=state)
    masks=out["masks"].detach().cpu().numpy().astype(np.uint8)
    if masks.ndim==4: masks=masks[:,0]
    scores=out["scores"].detach().float().cpu().numpy()
    np.savez_compressed(op,masks=masks,scores=scores)
print("DONE",flush=True)
''')
env=os.environ.copy(); env["HF_TOKEN"]=HF_TOKEN
cmd=[SAMPY,"-u",str(RUNNER),str(OUT/"candidate_manifest.csv"),str(RAW),str(BOX_SIZE_M),"1" if FORCE_SAM else "0"]
print("Starting targeted SAM3 delineation with live output...")
p=subprocess.run(cmd,env=env)
print("Return code:",p.returncode)
if p.returncode!=0: raise RuntimeError("SAM3 delineation subprocess failed; error printed above.")


## 5 — Convert SAM3 masks to georeferenced candidate crown polygons

For each candidate we retain the SAM3 mask component containing the U-Net peak (the chip centre), convert it to map coordinates, and attach the original U-Net peak probability. Implausibly small/large objects are discarded.


In [ ]:
pred=[]
with rasterio.open(RASTER) as src:
    for r in tqdm(manifest.itertuples(),total=len(manifest),desc="POLYGONIZE"):
        z=np.load(RAW/f"cand_{int(r.candidate_id):05d}.npz")
        masks=z["masks"]; scores=z["scores"]
        if len(masks)==0: continue
        # Prefer highest-score mask containing chip centre.
        order=np.argsort(scores)[::-1]; chosen=None
        for j in order:
            m=masks[j].astype(np.uint8)
            if m.shape!=(r.height,r.width):
                m=np.array(Image.fromarray(m).resize((r.width,r.height),resample=Image.Resampling.NEAREST))
            if m[r.height//2,r.width//2]>0: chosen=(m,float(scores[j])); break
        if chosen is None: continue
        m,score=chosen
        wt=rasterio.windows.transform(Window(r.col_off,r.row_off,r.width,r.height),src.transform)
        geoms=[]
        for gj,val in shapes(m,mask=m.astype(bool),transform=wt):
            if val==1: geoms.append(shape(gj))
        if not geoms: continue
        pt=Point(r.x,r.y)
        geom=max((q for q in geoms if q.contains(pt) or q.touches(pt)),key=lambda q:q.area,default=None)
        if geom is None: continue
        area=geom.area
        if not (MIN_PRED_AREA_M2<=area<=MAX_PRED_AREA_M2): continue
        pred.append(dict(candidate_id=int(r.candidate_id),peak_prob=float(r.peak_prob),
                         sam_score=score,crown_area_m2=area,geometry=geom))
crowns=gpd.GeoDataFrame(pred,crs=src.crs)
crowns.to_file(OUT/"Baego_UNet_guided_SAM3_crowns.gpkg",layer="crowns",driver="GPKG")
print("Candidates:",len(manifest))
print("Valid SAM3 crown objects:",len(crowns))
display(crowns.drop(columns="geometry").head())


## 6 — Add object-level U-Net probability summaries

The peak probability detects the candidate; here we also summarize the KVO_04 probability raster inside each SAM3 crown. These are useful object-level attributes for later filtering and satellite aggregation.


In [ ]:
from rasterio.mask import mask as rio_mask
rows=[]
with rasterio.open(PROB) as ps:
    for r in tqdm(crowns.itertuples(),total=len(crowns),desc="OBJECT PROBABILITY"):
        arr,_=rio_mask(ps,[r.geometry],crop=True,filled=False,indexes=1)
        vals=np.asarray(arr.compressed(),dtype=np.float32)
        vals=vals[vals!=65535]/10000.0
        rows.append(dict(candidate_id=r.candidate_id,
                         mean_prob=float(np.mean(vals)) if len(vals) else np.nan,
                         median_prob=float(np.median(vals)) if len(vals) else np.nan,
                         p90_prob=float(np.quantile(vals,.9)) if len(vals) else np.nan))
stats=pd.DataFrame(rows)
crowns=crowns.merge(stats,on="candidate_id",how="left")
crowns.to_file(OUT/"Baego_UNet_guided_SAM3_crowns.gpkg",layer="crowns",driver="GPKG")
print(crowns[["peak_prob","mean_prob","median_prob","crown_area_m2"]].describe())


## 7 — Primary observational validation against Crown_Kristof

`Crown_Kristof.shp` contains **all visible crowns digitized within the six areas of `ROI_Crowns_Kvo.shp`**. This is therefore the primary observational basis for KVO_06.

The analysis is restricted to the six ROIs and reports both directions:

- **Reference recovery:** for every digitized visible crown, the best overlapping automatic U-Net→SAM3 object and its IoU.
- **Automatic-object matching:** for every automatic crown, the best overlapping digitized crown and its IoU.

This allows unmatched automatic objects *inside these exhaustively digitized visible-crown areas* to be identified explicitly, while keeping the geometric match statistics separate from the U-Net species-probability evidence.


In [ ]:
from shapely.validation import make_valid

def safe_iou(a,b):
    try:
        a=make_valid(a); b=make_valid(b)
        inter=a.intersection(b).area
        if inter<=0: return 0.0
        union=a.union(b).area
        return float(inter/union) if union>0 else 0.0
    except Exception:
        return 0.0

ref=gpd.read_file(CROWN_REF)
roi=gpd.read_file(CROWN_ROI)
if ref.crs is None:
    print("Assigning EPSG:32635 to Crown_Kristof (Baego digitizing CRS)")
    ref=ref.set_crs("EPSG:32635")
if roi.crs is None: raise ValueError("ROI_Crowns_Kvo missing CRS")
if ref.crs!=crowns.crs: ref=ref.to_crs(crowns.crs)
if roi.crs!=crowns.crs: roi=roi.to_crs(crowns.crs)
ref=ref[ref.geometry.notna() & ~ref.geometry.is_empty].copy()
roi=roi[roi.geometry.notna() & ~roi.geometry.is_empty].copy()
ref["geometry"]=ref.geometry.apply(make_valid)
roi["geometry"]=roi.geometry.apply(make_valid)
roi_union=roi.geometry.union_all()

# Enforce the observational footprint for both datasets.
ref=ref[ref.geometry.intersects(roi_union)].copy().reset_index(drop=True)
auto=crowns[crowns.geometry.intersects(roi_union)].copy().reset_index(drop=True)
print("Validation ROIs:",len(roi))
print("Digitized visible reference crowns:",len(ref))
print("Automatic SAM3 crowns in ROIs:",len(auto))

# Automatic object -> best reference crown.
ref_sidx=ref.sindex
auto_rows=[]
for r in tqdm(auto.itertuples(),total=len(auto),desc="AUTO -> REFERENCE"):
    ids=list(ref_sidx.query(r.geometry,predicate="intersects"))
    pairs=[(int(i),safe_iou(r.geometry,ref.iloc[i].geometry)) for i in ids]
    bi,bv=max(pairs,key=lambda z:z[1],default=(-1,0.0))
    auto_rows.append(dict(candidate_id=int(r.candidate_id),best_ref_index=bi,best_ref_iou=bv))
auto_qa=pd.DataFrame(auto_rows)
auto_qa.to_csv(OUT/"validation_auto_to_reference.csv",index=False)

# Reference crown -> best automatic object.
auto_sidx=auto.sindex
ref_rows=[]
for i,r in tqdm(ref.iterrows(),total=len(ref),desc="REFERENCE -> AUTO"):
    ids=list(auto_sidx.query(r.geometry,predicate="intersects"))
    pairs=[(int(j),safe_iou(r.geometry,auto.iloc[j].geometry)) for j in ids]
    bj,bv=max(pairs,key=lambda z:z[1],default=(-1,0.0))
    cid=int(auto.iloc[bj].candidate_id) if bj>=0 else -1
    ref_rows.append(dict(ref_index=int(i),best_candidate_id=cid,best_auto_iou=bv))
ref_qa=pd.DataFrame(ref_rows)
ref_qa.to_csv(OUT/"validation_reference_to_auto.csv",index=False)

nref=len(ref); nauto=len(auto)
print("\nREFERENCE RECOVERY")
print("Any overlap:",int((ref_qa.best_auto_iou>0).sum()),"/",nref)
print("IoU >= 0.5:",int((ref_qa.best_auto_iou>=0.5).sum()),"/",nref)
print("IoU >= 0.7:",int((ref_qa.best_auto_iou>=0.7).sum()),"/",nref)
if (ref_qa.best_auto_iou>0).any():
    print("Median IoU among overlapping reference crowns:",
          float(ref_qa.loc[ref_qa.best_auto_iou>0,"best_auto_iou"].median()))

print("\nAUTOMATIC OBJECT MATCHING")
print("Any reference overlap:",int((auto_qa.best_ref_iou>0).sum()),"/",nauto)
print("IoU >= 0.5:",int((auto_qa.best_ref_iou>=0.5).sum()),"/",nauto)
print("IoU >= 0.7:",int((auto_qa.best_ref_iou>=0.7).sum()),"/",nauto)
if (auto_qa.best_ref_iou>0).any():
    print("Median IoU among overlapping automatic crowns:",
          float(auto_qa.loc[auto_qa.best_ref_iou>0,"best_ref_iou"].median()))


## 8 — Diagnostics and production summary

In [ ]:
fig,ax=plt.subplots(figsize=(7,5))
if len(crowns):
    ax.scatter(crowns.peak_prob,crowns.mean_prob,s=12,alpha=.6)
ax.set_xlabel("U-Net candidate peak probability"); ax.set_ylabel("Mean U-Net probability inside SAM3 crown")
ax.set_title("Baego: detector → delineator consistency")
plt.show()

summary={
    "probability_threshold":PROB_THRESHOLD,
    "minimum_candidate_separation_m":PEAK_MIN_DISTANCE_M,
    "n_candidates":int(len(manifest)),
    "n_valid_sam3_crowns":int(len(crowns)),
    "median_crown_area_m2":float(crowns.crown_area_m2.median()) if len(crowns) else None,
    "median_object_mean_probability":float(crowns.mean_prob.median()) if len(crowns) else None,
}
(OUT/"summary.json").write_text(json.dumps(summary,indent=2))
print(json.dumps(summary,indent=2))


## 9 — Interpretation / next step

KVO_06 now uses the six exhaustively digitized crown areas as its **observational basis**.

- **KVO_03:** independent leave-one-site-out evidence for U-Net species classification.
- **KVO_04:** final continuous Musanga probability maps.
- **KVO_05:** prompt-conditioned SAM3 crown-boundary benchmark.
- **KVO_06:** fully automatic U-Net → SAM3 workflow evaluated inside six areas where all visible crowns were digitized.

The production-scale Baego inventory should only be expanded after examining recovery, matching IoU, and unmatched automatic objects in these six validation areas.


## 10 — Experiment A: adaptive U-Net-guided SAM3 prompts

Uses local 0.60 probability components for approximate size and four prompt bins (10/14/18/22m). The original 0.90 candidate threshold is retained for seeds. This is a first sensitivity experiment, not final calibration.


In [ ]:

# BLOCK 10 — Adaptive U-Net box prompts; reuses existing chips and SAM3 runner.
from scipy.ndimage import label as ndi_label
from shapely.validation import make_valid
EXP=OUT/"experiments_v09"; EXP.mkdir(exist_ok=True,parents=True)
MAN=pd.read_csv(OUT/"candidate_manifest.csv")
BOX_VALUES=[10.,14.,18.,22.]
def box_from_probability(r,threshold=.60,margin=1.25):
    with rasterio.open(PROB) as ds:
        rr,cc=ds.index(float(r.x),float(r.y))
        rad=int(round(15/abs(ds.transform.a)))
        a=ds.read(1,window=Window(cc-rad,rr-rad,2*rad+1,2*rad+1),boundless=True,fill_value=65535)
        good=(a!=65535)&(a>=int(threshold*10000))
        labs,num=ndi_label(good)
        k=labs[rad,rad]
        if k==0:return 14.
        ys,xs=np.where(labs==k)
        extent=max((xs.max()-xs.min()+1)*abs(ds.transform.a),(ys.max()-ys.min()+1)*abs(ds.transform.e))
        return float(np.clip(extent*margin,8,22))
MAN["adaptive_box_m"]=[box_from_probability(r) for r in MAN.itertuples()]
MAN.to_csv(EXP/"adaptive_boxes.csv",index=False)
print(MAN.adaptive_box_m.describe())
# Existing runner accepts one box size per call; group adaptive boxes into 4 bins,
# retaining original candidate_id and output paths for reproducibility.
MAN["box_group"]=MAN.adaptive_box_m.apply(lambda x:min(BOX_VALUES,key=lambda b:abs(b-x)))
adaptive_raw=EXP/"adaptive_raw";adaptive_raw.mkdir(exist_ok=True)
for b,g in MAN.groupby("box_group"):
    mf=EXP/f"adaptive_manifest_{int(b)}m.csv";g.to_csv(mf,index=False)
    p=subprocess.run([SAMPY,"-u",str(RUNNER),str(mf),str(adaptive_raw),str(b),"0"],env=env)
    if p.returncode:raise RuntimeError(f"SAM3 failed for {b} m")
print("Adaptive masks ready:",len(list(adaptive_raw.glob("cand_*.npz"))))


## 11 — Adaptive crown agreement and reproducibility

Compare against manual interpreted boundaries; no species-level validation claim.


In [ ]:

# BLOCK 11 — Shared polygonization and manual–automatic agreement.
from rasterio.features import shapes
from shapely.geometry import Point,shape
from PIL import Image
def polygons_from_masks(manifest_df,raw_dir):
    records=[]
    with rasterio.open(RASTER) as src:
        for r in tqdm(manifest_df.itertuples(),total=len(manifest_df),desc="POLYGONIZE"):
            p=raw_dir/f"cand_{int(r.candidate_id):05d}.npz"
            if not p.exists():continue
            z=np.load(p);masks=z["masks"];scores=z["scores"]
            if not len(masks):continue
            for j in np.argsort(scores)[::-1]:
                m=masks[j].astype("uint8")
                if m.ndim==3:m=m[0]
                if m.shape!=(r.height,r.width):
                    m=np.array(Image.fromarray(m).resize((r.width,r.height),Image.Resampling.NEAREST))
                if not m[r.height//2,r.width//2]:continue
                wt=rasterio.windows.transform(Window(r.col_off,r.row_off,r.width,r.height),src.transform)
                pt=Point(r.x,r.y)
                geoms=[shape(g) for g,v in shapes(m,mask=m.astype(bool),transform=wt) if v==1]
                geom=max((g for g in geoms if g.intersects(pt)),key=lambda g:g.area,default=None)
                if geom is not None and MIN_PRED_AREA_M2<=geom.area<=MAX_PRED_AREA_M2:
                    records.append(dict(candidate_id=int(r.candidate_id),area_m2=geom.area,geometry=make_valid(geom)))
                break
    return gpd.GeoDataFrame(records,geometry="geometry",crs=rasterio.open(RASTER).crs)
def agreement(auto,reference):
    if auto.crs!=reference.crs:auto=auto.to_crs(reference.crs)
    idx=reference.sindex;rows=[]
    for a in auto.itertuples():
        best=(-1,0.,0.,np.nan)
        for j in idx.query(a.geometry,predicate="intersects"):
            g=reference.iloc[j].geometry
            inter=a.geometry.intersection(g).area
            union=a.geometry.union(g).area
            iou=inter/union if union else 0.
            if iou>best[1]:best=(int(j),iou,inter,g.area)
        j,iou,inter,refarea=best
        rows.append(dict(candidate_id=a.candidate_id,manual_index=j,iou=iou,area_ratio=a.geometry.area/refarea if j>=0 else np.nan,manual_coverage=inter/refarea if j>=0 else np.nan,sam_inside_manual=inter/a.geometry.area if a.geometry.area else np.nan))
    return pd.DataFrame(rows)
reference=gpd.read_file(CROWN_REF)
if reference.crs is None:reference=reference.set_crs("EPSG:32635")
reference=reference[reference.geometry.intersects(roi_union)].copy()
adaptive_crowns=polygons_from_masks(MAN,adaptive_raw)
adaptive_qa=agreement(adaptive_crowns,reference)
adaptive_qa.to_csv(EXP/"adaptive_agreement.csv",index=False)
adaptive_crowns.to_file(EXP/"adaptive_crowns.gpkg",driver="GPKG")
print("Adaptive valid crowns:",len(adaptive_crowns),"overlap:",(adaptive_qa.iou>0).sum(),"median matched IoU:",adaptive_qa.loc[adaptive_qa.iou>0,"iou"].median())
# Reproducibility: compare the original 10m mask files against the sensitivity-run 10m files.
original_ids={int(p.stem.split("_")[-1]) for p in RAW.glob("cand_*.npz")}
print("Original 10m raw masks:",len(original_ids),"adaptive masks:",len(list(adaptive_raw.glob("cand_*.npz"))))


## 12 — Experiment B: species-independent RGB crown candidates

**Pilot approximation:** RGB local-extrema proposals followed by SAM3 geometric prompts. This is not native SAM3 automatic discovery and may miss crowns. The 846 manual crowns are observational comparisons only.


In [ ]:

# BLOCK 12 — Species-independent crown proposals within the six ROIs.
# Generic RGB candidate generation via local canopy brightness/texture extrema.
# This is a pilot heuristic, NOT native automatic SAM3 discovery; compare empirically.
from scipy.ndimage import gaussian_filter,maximum_filter
from rasterio.mask import mask as rio_mask
from shapely.geometry import mapping
generic=[]
with rasterio.open(RASTER) as src:
    for k,region in enumerate(roi.geometry):
        arr,tr=rio_mask(src,[mapping(region)],crop=True,indexes=[1,2,3],filled=True,nodata=0)
        rgb=arr.astype("float32").mean(axis=0)
        # Local RGB texture peaks, spaced ~5m; exclude masked background.
        px=abs(src.transform.a); sep=max(3,int(round(5/px)))
        smooth=gaussian_filter(rgb,sigma=max(2,sep/5))
        peak=(smooth==maximum_filter(smooth,size=2*sep+1))&(rgb>20)
        ys,xs=np.where(peak)
        for y,x in zip(ys,xs):
            xx,yy=rasterio.transform.xy(tr,int(y),int(x))
            if region.covers(Point(xx,yy)):generic.append((k,xx,yy))
print("Generic RGB candidates:",len(generic))
# Build independent chips, same manifest contract as Block 4.
grows=[]
with rasterio.open(RASTER) as src:
    px=abs(src.transform.a);size=max(128,int(round(CHIP_SIZE_M/px)));size+=size%2
    for i,(region,x,y) in enumerate(tqdm(generic,desc="GENERIC RGB CHIPS")):
        row,col=src.index(x,y);w=Window(col-size//2,row-size//2,size,size)
        out=src.read([1,2,3],window=w,boundless=True,fill_value=0).transpose(1,2,0)
        path=EXP/f"generic_{i:05d}.png"
        if not path.exists():Image.fromarray(out.astype("uint8")).save(path)
        grows.append(dict(candidate_id=i,region=region,x=x,y=y,row_off=int(w.row_off),col_off=int(w.col_off),width=size,height=size,gsd=px,path=str(path)))
generic_manifest=pd.DataFrame(grows)
generic_manifest.to_csv(EXP/"generic_manifest.csv",index=False)
print("Generic chips:",len(generic_manifest))


## 13 — Generic RGB-guided SAM3 and U-Net attribution


In [ ]:

# BLOCK 13 — Species-independent SAM3 segmentation + agreement.
# Uses 14m generic prompts; independent of U-Net seed threshold.
generic_raw=EXP/"generic_raw";generic_raw.mkdir(exist_ok=True)
p=subprocess.run([SAMPY,"-u",str(RUNNER),str(EXP/"generic_manifest.csv"),str(generic_raw),"14","0"],env=env)
if p.returncode:raise RuntimeError("Generic SAM3 failed")
generic_crowns=polygons_from_masks(generic_manifest,generic_raw)
generic_qa=agreement(generic_crowns,reference)
generic_qa.to_csv(EXP/"generic_agreement.csv",index=False)
generic_crowns.to_file(EXP/"generic_crowns.gpkg",driver="GPKG")
print("Generic valid:",len(generic_crowns),"overlapping manual:",int((generic_qa.iou>0).sum()),"median matched IoU:",generic_qa.loc[generic_qa.iou>0,"iou"].median())
# Overlay species probability only AFTER delineation.
from rasterio.mask import mask as rio_mask
with rasterio.open(PROB) as ps:
    vals=[]
    for g in generic_crowns.geometry:
        a,_=rio_mask(ps,[mapping(g)],crop=True,indexes=1,filled=False)
        v=np.asarray(a.compressed(),dtype="float32");v=v[v!=65535]/10000
        vals.append(float(v.mean()) if len(v) else np.nan)
generic_crowns["musanga_mean_prob"]=vals
generic_crowns.to_file(EXP/"generic_crowns_with_probability.gpkg",driver="GPKG")
display(pd.DataFrame([{"method":"adaptive","valid":len(adaptive_crowns),"matched":int((adaptive_qa.iou>0).sum()),"median_iou":adaptive_qa.loc[adaptive_qa.iou>0,"iou"].median()},{"method":"generic_rgb","valid":len(generic_crowns),"matched":int((generic_qa.iou>0).sum()),"median_iou":generic_qa.loc[generic_qa.iou>0,"iou"].median()}]))


## 14 — Revised all-crown reference and Baego DSM audit

Run **0R** first in a new session; **Blocks 14–16 do not need SAM3, GPU, or Blocks 1–13**. This reads the revised Crown_Kristof shapefile afresh and checks DSM/RGB alignment.


In [ ]:
# BLOCK 14 — Audit revised all-crown reference and DSM (no SAM3/GPU)
from rasterio.warp import WarpedVRT,Resampling
from rasterio.mask import mask as rio_mask
from shapely.geometry import mapping
from scipy.ndimage import gaussian_filter,maximum_filter,minimum_filter
from shapely.validation import make_valid
DSM=GEO/"Baego_DSM.tif"
assert DSM.exists(),f"DSM not found: {DSM}"
reference_v2=gpd.read_file(CROWN_REF)
if reference_v2.crs is None:
    print("Crown_Kristof CRS absent: assigning EPSG:32635 (verify digitizing CRS)")
    reference_v2=reference_v2.set_crs("EPSG:32635")
with rasterio.open(RASTER) as rgb,rasterio.open(DSM) as dsm:
    reference_v2=reference_v2.to_crs(rgb.crs)
    roi_v2=gpd.read_file(CROWN_ROI).to_crs(rgb.crs)
    print("RGB:",rgb.crs,rgb.res,"DSM:",dsm.crs,dsm.res)
    print("DSM aligned:",rgb.crs==dsm.crs and rgb.transform==dsm.transform and rgb.shape==dsm.shape)
reference_v2=reference_v2[reference_v2.geometry.notna()&(~reference_v2.geometry.is_empty)].copy()
reference_v2["geometry"]=reference_v2.geometry.map(make_valid)
ref_roi_union=roi_v2.geometry.union_all()
reference_v2=reference_v2[reference_v2.geometry.intersects(ref_roi_union)].copy()
print("Revised crowns:",len(reference_v2),"ROIs:",len(roi_v2),"median area m2:",round(reference_v2.geometry.area.median(),1))
print("Invalid after repair:",(~reference_v2.is_valid).sum())
print("Per-ROI counts:",[int(reference_v2.intersects(g).sum()) for g in roi_v2.geometry])


## 15 — Species-independent DSM + RGB candidate fusion

DSM canopy maxima complement the existing 600 RGB candidates. DSM is locally detrended; this is a proposal experiment, not validated crown height or a terrain-normalized CHM. Keep original RGB candidates and add non-duplicate DSM peaks.


In [ ]:
# BLOCK 15 — DSM treetop candidates, merged with existing RGB candidates
# Host-only, no SAM3; DSM is elevation, not CHM. Use local relief and smoothing.
from scipy.ndimage import gaussian_filter,maximum_filter,minimum_filter
from rasterio.warp import WarpedVRT,Resampling
from rasterio.mask import mask as rio_mask
from shapely.geometry import mapping,Point
from scipy.spatial import cKDTree
DSM_PEAK_SEPARATION_M=4.0
DSM_SMOOTH_M=0.7
DSM_RELIEF_RADIUS_M=12.0
DSM_MIN_LOCAL_RELIEF_M=1.0  # diagnostic parameter; adjust after inspecting coverage
candidate_rows=[]
with rasterio.open(RASTER) as rgb,rasterio.open(DSM) as dsm:
    with WarpedVRT(dsm,crs=rgb.crs,transform=rgb.transform,width=rgb.width,height=rgb.height,resampling=Resampling.bilinear) as aligned:
        gsd=abs(rgb.transform.a)
        for roi_id,geom in enumerate(roi_v2.geometry):
            z,tr=rio_mask(aligned,[mapping(geom)],crop=True,indexes=1,filled=False)
            a=np.asarray(z.filled(np.nan),dtype="float32")
            valid=np.isfinite(a)&(~np.ma.getmaskarray(z))
            if not valid.any():continue
            fill=float(np.nanmedian(a[valid]))
            sm=gaussian_filter(np.where(valid,a,fill),sigma=max(1,DSM_SMOOTH_M/gsd))
            local_base=minimum_filter(sm,size=2*max(1,int(DSM_RELIEF_RADIUS_M/gsd))+1)
            relief=sm-local_base
            radius=max(1,int(round(DSM_PEAK_SEPARATION_M/gsd)))
            peaks=(sm==maximum_filter(sm,size=2*radius+1))&valid&(relief>=DSM_MIN_LOCAL_RELIEF_M)
            ys,xs=np.where(peaks)
            for y,x in zip(ys,xs):
                xx,yy=rasterio.transform.xy(tr,int(y),int(x))
                if geom.covers(Point(xx,yy)):
                    candidate_rows.append(dict(source="DSM",roi=roi_id,x=xx,y=yy,local_relief_m=float(relief[y,x])))
dsm_points=pd.DataFrame(candidate_rows)
print("DSM peaks:",len(dsm_points))
rgb_points=generic_manifest[["x","y"]].copy();rgb_points["source"]="RGB";rgb_points["roi"]=-1;rgb_points["local_relief_m"]=np.nan
combined=pd.concat([rgb_points,dsm_points],ignore_index=True)
# Preserve RGB points; add DSM peaks only when >3m from existing candidate.
accepted=rgb_points.copy()
if len(dsm_points):
    tree=cKDTree(accepted[["x","y"]].to_numpy())
    for r in dsm_points.itertuples():
        if tree.query([r.x,r.y])[0]>=3.0:
            accepted=pd.concat([accepted,pd.DataFrame([dict(x=r.x,y=r.y,source="DSM",roi=r.roi,local_relief_m=r.local_relief_m)])],ignore_index=True)
            tree=cKDTree(accepted[["x","y"]].to_numpy())
accepted["candidate_id"]=np.arange(len(accepted))
accepted.to_csv(OUT/"experiments_v09"/"rgb_dsm_candidate_points.csv",index=False)
print("Fused candidates:",len(accepted))


## 16 — Candidate coverage against updated all-crown reference

Report new manual crown count and candidate coverage before expensive SAM3 processing.


In [ ]:
# BLOCK 16 — Coverage comparison against revised ALL-crown reference
from shapely.geometry import Point
from geopandas import sjoin
def coverage(points,ref):
    p=gpd.GeoDataFrame(points.copy(),geometry=gpd.points_from_xy(points.x,points.y),crs=ref.crs)
    j=gpd.sjoin(p,ref[["geometry"]].reset_index(drop=True),how="left",predicate="within")
    matched=j.index_right.dropna().astype(int)
    return dict(candidates=len(p),crowns=len(ref),crowns_covered=matched.nunique(),coverage_pct=round(100*matched.nunique()/len(ref),1),candidates_inside=int(j.loc[j.index_right.notna()].index.nunique()),candidates_outside=int(len(p)-j.loc[j.index_right.notna()].index.nunique()))
display(pd.DataFrame([dict(method="RGB",**coverage(rgb_points,reference_v2)),dict(method="RGB + DSM",**coverage(accepted,reference_v2))]))
print("Diagnostic only. Inspect DSM peaks on RGB before launching new SAM3 inference.")
